# AirShift — Model Development & Comparison

## 1. Load the Labeled Dataset

The final labeled dataset created in the previous stage is loaded as the starting point for model development.

The dataset contains the engineered air quality and meteorological features together with the binary `Deterioration` target.

The labeled dataset is used without modifying the original saved file. Subsequent preprocessing and model development steps are performed on working copies of the data.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [5]:
# Define the data directories.
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")

# Load the final labeled dataset created in the labeling stage.
labeled_file = DATA_DIR / "airshift_labeled.csv"

df = pd.read_csv(
    labeled_file,
    parse_dates=["datetime"]
)

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print(
    "Date range:",
    df["datetime"].min(),
    "to",
    df["datetime"].max()
)

Dataset shape: (418381, 100)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 17:00:00


In [6]:
# Check the final target distribution before model development.

print("Target distribution:")
print(df["Deterioration"].value_counts())

print("\nTarget distribution (%):")
print(
    df["Deterioration"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target distribution:
Deterioration
0.0    219985
1.0    198396
Name: count, dtype: int64

Target distribution (%):
Deterioration
0.0    52.58
1.0    47.42
Name: proportion, dtype: float64


## 2. Define Features and Target

The dataset is separated into input features and the target variable required for supervised machine learning.

The `Deterioration` column is used as the binary target, while the remaining relevant variables are considered candidate input features.

Identifiers and columns that do not provide predictive information are excluded from the feature set.

The feature set will be further examined in the following leakage-audit stage before training the models.


In [7]:
# Define the target variable for binary classification.
TARGET = "Deterioration"

# Columns that should not be used as model features.
# "No" is an observation identifier, while "datetime" is handled
# separately for temporal splitting and is not directly used as a feature.
EXCLUDED_COLUMNS = [
    TARGET,
    "No",
    "datetime"
]

# Create the feature matrix and target vector.
X = df.drop(
    columns=EXCLUDED_COLUMNS
)

y = df[TARGET].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget values:")
print(sorted(y.unique()))

print("\nNumber of features:", X.shape[1])

Feature matrix shape: (418381, 97)
Target shape: (418381,)

Target values:
[np.int64(0), np.int64(1)]

Number of features: 97


## 3. Leakage Audit

Before training the machine learning models, the candidate features are examined for potential data leakage.

Data leakage occurs when a model receives information that would not have been available at the prediction time.

Because AirShift is an early-warning system, all model features must represent information available at or before the prediction time. Future-derived variables used during target construction must therefore be excluded from the feature set.

This audit checks the feature names for future-related information and confirms that the target variable is not included among the input features.


In [8]:
# Identify candidate features that may contain future-derived information.
# These features must not be used as model inputs.

future_keywords = [
    "future",
    "Deterioration"
]

potential_leakage_features = [
    column
    for column in X.columns
    if any(keyword.lower() in column.lower() for keyword in future_keywords)
]

print("Potential leakage features:")
print(potential_leakage_features)

Potential leakage features:
[]


In [9]:
# Confirm that the target variable is not present in the feature matrix.

print("Target included in X:", TARGET in X.columns)

Target included in X: False


In [10]:
# Check the dataset for any future-derived columns that should not
# be available as model features.

future_columns_in_dataset = [
    column
    for column in df.columns
    if "future" in column.lower()
]

print("Future-derived columns in dataset:")
print(future_columns_in_dataset)

Future-derived columns in dataset:
[]


## 4. Temporal Train/Test Split

Because AirShift is designed to predict future air quality deterioration, the dataset is divided chronologically rather than randomly.

Earlier observations are used for model training, while later observations are reserved for testing.

This approach reflects the real-world prediction setting, where a model learns from historical observations and is then applied to future observations.

A temporal split also reduces the risk of information from future periods influencing model development.


In [11]:
# Define a chronological train/test split.
# Earlier years are used for training and later years are reserved
# for evaluating how well the models generalize to future observations.

TRAIN_END_DATE = "2015-12-31 23:00:00"
TEST_START_DATE = "2016-01-01 00:00:00"

train_mask = df["datetime"] <= TRAIN_END_DATE
test_mask = df["datetime"] >= TEST_START_DATE

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Training period:")
print(
    train_df["datetime"].min(),
    "to",
    train_df["datetime"].max()
)

print("\nTesting period:")
print(
    test_df["datetime"].min(),
    "to",
    test_df["datetime"].max()
)

print("\nTraining observations:", len(train_df))
print("Testing observations:", len(test_df))

Training period:
2013-03-01 00:00:00 to 2015-12-31 23:00:00

Testing period:
2016-01-01 00:00:00 to 2017-02-28 17:00:00

Training observations: 296481
Testing observations: 121900


In [12]:
# Separate features and target for the training and testing datasets.

X_train = train_df.drop(
    columns=EXCLUDED_COLUMNS
)

y_train = train_df[TARGET].astype(int)

X_test = test_df.drop(
    columns=EXCLUDED_COLUMNS
)

y_test = test_df[TARGET].astype(int)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (296481, 97)
y_train shape: (296481,)
X_test shape: (121900, 97)
y_test shape: (121900,)


In [13]:
# Verify that there is no temporal overlap between training and testing data.

print("Latest training observation:")
print(train_df["datetime"].max())

print("\nEarliest testing observation:")
print(test_df["datetime"].min())

print(
    "\nTemporal overlap:",
    train_df["datetime"].max() >= test_df["datetime"].min()
)

Latest training observation:
2015-12-31 23:00:00

Earliest testing observation:
2016-01-01 00:00:00

Temporal overlap: False


### Temporal Split Findings

The dataset was divided chronologically to reflect the real-world deployment scenario of the AirShift early-warning system.

The training set contains **296,481 observations** covering the period from **March 2013 to December 2015**, while the test set contains **121,900 observations** covering **January 2016 to February 2017**.

No temporal overlap exists between the two datasets. This ensures that the models are trained exclusively on historical observations and evaluated on later, unseen observations.


## 5. Prepare Features for Model Training

The training and testing features contain both numerical and categorical variables.

The categorical variables, `station` and `wd`, are converted into numerical representations using one-hot encoding. The encoder is fitted only on the training data and then applied to the test data to prevent data leakage.

Numerical features are kept unchanged at this stage. Scaling will be applied only where required by the individual machine learning model.

This preprocessing structure allows the three candidate models to be evaluated using the same underlying feature information while applying model-specific preprocessing when necessary.


In [14]:
# Identify categorical and numerical features.
# Categorical features require encoding before model training.

categorical_features = [
    "station",
    "wd"
]

numerical_features = [
    column
    for column in X_train.columns
    if column not in categorical_features
]

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

print("\nCategorical features:")
print(categorical_features)

Number of categorical features: 2
Number of numerical features: 95

Categorical features:
['station', 'wd']


In [15]:
# Create a preprocessing transformer for categorical variables.
# The encoder will be fitted only on the training data.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessing transformer created successfully.")

Preprocessing transformer created successfully.


## 6. Define the Three Candidate Models

Three machine learning models are selected for the initial model comparison.

The models represent different approaches to binary classification:

* **Logistic Regression** provides a simple and interpretable baseline model.
* **Random Forest** captures non-linear relationships and interactions between features.
* **XGBoost** provides a gradient boosting approach that is effective for structured tabular data.

The models are initially evaluated using reasonable default configurations without extensive hyperparameter tuning.

Hyperparameter tuning will be performed only for the best-performing model in the following model tuning stage.


### 6.1 Import Model Libraries

The required machine learning algorithms and preprocessing components are imported for model development.

Logistic Regression, Random Forest, and XGBoost will be evaluated using the same training and validation data.

They were added to the first cell (loading section)

### 6.2 Logistic Regression

Logistic Regression is used as the baseline classification model.

Because Logistic Regression is sensitive to feature scale, numerical features will be standardized before model training.

Missing numerical values are handled using training-data statistics, while categorical variables are encoded using one-hot encoding.

All preprocessing is performed inside the pipeline to ensure that information from future data is not used during training.


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        )
    ]
)

logistic_pipeline = Pipeline([
    ("preprocessor", logistic_preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Logistic Regression pipeline created successfully.")

Logistic Regression pipeline created successfully.


### 6.3 Random Forest

Random Forest is included to capture non-linear relationships between air quality, meteorological, temporal, and historical pollution features.

Unlike Logistic Regression, Random Forest does not require feature scaling.

Numerical missing values are imputed using the training data, while categorical variables are imputed and one-hot encoded.


In [17]:
random_forest_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numerical_features
        )
    ]
)

random_forest_pipeline = Pipeline([
    ("preprocessor", random_forest_preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


### 6.4 XGBoost

XGBoost is included as a gradient boosting model for structured tabular data.

It can capture complex non-linear relationships and interactions between the engineered air quality and meteorological features.

Feature scaling is not required for XGBoost, so numerical features are imputed without standardization.

The initial configuration is intentionally kept simple because hyperparameter tuning will be performed separately for the best-performing model.


In [18]:
xgboost_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numerical_features
        )
    ]
)

xgboost_pipeline = Pipeline([
    ("preprocessor", xgboost_preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss"
        )
    )
])

print("XGBoost pipeline created successfully.")

XGBoost pipeline created successfully.


### 6.5 Store Candidate Models

The three candidate pipelines are stored in a single dictionary to simplify model training, prediction, and evaluation.

Each pipeline contains its own preprocessing steps and machine learning model.


In [19]:
models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
    "XGBoost": xgboost_pipeline
}

print("Candidate models:")
for name in models:
    print("-", name)

Candidate models:
- Logistic Regression
- Random Forest
- XGBoost


In [20]:
print("Number of candidate models:", len(models))

Number of candidate models: 3


## 7. Temporal Validation Split

A validation set is created from the original training period to compare the three candidate models without using the final test data.

Because AirShift is a time-dependent early-warning system, the validation split is performed chronologically.

Observations from **2013 to 2014** are used for model training, while observations from **2015** are used for validation.

The previously reserved **2016–2017** period remains untouched and will be used only for the final evaluation of the selected model.

This approach prevents the final test period from influencing model selection.


7.1 Define the Validation Period

In [21]:
VALIDATION_START_DATE = "2015-01-01 00:00:00"
VALIDATION_END_DATE = "2015-12-31 23:00:00"

model_train_mask = (
    train_df["datetime"] < VALIDATION_START_DATE
)

validation_mask = (
    (train_df["datetime"] >= VALIDATION_START_DATE)
    & (train_df["datetime"] <= VALIDATION_END_DATE)
)

model_train_df = train_df.loc[model_train_mask].copy()
validation_df = train_df.loc[validation_mask].copy()

print("Model training period:")
print(
    model_train_df["datetime"].min(),
    "to",
    model_train_df["datetime"].max()
)

print("\nValidation period:")
print(
    validation_df["datetime"].min(),
    "to",
    validation_df["datetime"].max()
)

print("\nModel training observations:", len(model_train_df))
print("Validation observations:", len(validation_df))

Model training period:
2013-03-01 00:00:00 to 2014-12-31 23:00:00

Validation period:
2015-01-01 00:00:00 to 2015-12-31 23:00:00

Model training observations: 191665
Validation observations: 104816
